In [ ]:
import numpy as np
from freshflow.loader import load_sales
from freshflow.validation import column, invalid_masks, missing_masks, rule_counts

data = load_sales("../data/raw/freshflow_sales.csv")

In [ ]:
missing = rule_counts(missing_masks(data))
for name, count in missing["per_rule"].items():
    print(f"{name:28} {count}")

In [ ]:
masks = invalid_masks(data)
result = rule_counts(masks)

for name, count in result["per_rule"].items():
    print(f"{name:28} {count}")

print("Total rule violations:", result["total_violations"])
print("Unique rows affected:", result["unique_rows_affected"])

In [ ]:
targets = {
    "duplicate transaction_id": 5,
    "quantity not positive": 4,
    "price not positive": 2,
    "negative unit cost": 2,
    "discount outside 0-1": 2,
    "invalid rating": 3,
    "category mismatch": 4,
    "unknown outlet": 2,
}

for name, target in targets.items():
    print(f"{name:28} target={target}  found={result['per_rule'][name]}")

In [ ]:
bad_quantity = np.nonzero(masks["quantity not positive"])[0]
print(bad_quantity)
print(data[bad_quantity])

In [ ]:
duplicate_rows = np.nonzero(masks["duplicate transaction_id"])[0]
subset = data[duplicate_rows]
print(subset[np.argsort(subset[:, 0])])

In [ ]:
quantity = column(data, "quantity")
quantity_status = np.where(quantity > 0, "valid", "invalid")
print(np.unique(quantity_status, return_counts=True))

discount = column(data, "discount_rate")
band = np.select(
    [discount > 1, discount > 0.20, discount > 0],
    ["impossible", "high", "normal"],
    default="none",
)
print(np.unique(band, return_counts=True))

In [ ]:
outlet = column(data, "outlet_id")
category = column(data, "category_code")

high_discount = data[(discount > 0.20) & (discount <= 1)]
outlet_b_juice = data[(outlet == 2) & (category == 1)]
not_outlet_a = data[~(outlet == 1)]

print(len(high_discount), len(outlet_b_juice), len(not_outlet_a))

In [ ]:
from freshflow.loader import save_array, COLUMNS
from freshflow.cleaning import clean_data, cleaning_report, write_cleaning_report

raw_copy = data.copy()
clean, excluded, masks, exclude = clean_data(data)

print("Raw rows:", len(data))
print("Excluded rows:", len(excluded))
print("Clean rows:", len(clean))

In [ ]:
for line in cleaning_report(masks, exclude):
    print(line)

In [ ]:
print(data[masks["quantity not positive"]])

In [ ]:
duplicates = data[masks["duplicate transaction_id"]]
print(duplicates[np.argsort(duplicates[:, 0])])
print("Duplicate-ID rows still in clean data:", np.isin(duplicates[:, 0], clean[:, 0]).sum())

In [ ]:
print("Raw unchanged:", np.array_equal(data, raw_copy, equal_nan=True))
print("Clean is a separate copy:", not np.shares_memory(data, clean))

In [ ]:
for name in ["unit_price", "unit_cost", "customer_rating"]:
    print(name, np.isnan(column(clean, name)).sum())

In [ ]:
save_array("../data/processed/freshflow_clean.csv", clean)
save_array("../data/processed/freshflow_invalid_records.csv", excluded, COLUMNS + ["reason_code"])
write_cleaning_report("../data/processed/cleaning_report.txt", masks, exclude, len(data))